# SIV: Systematic Inference-Based Verification for NL-to-FOL Translation

End-to-end pipeline notebook.  Works on Google Colab and locally.

**Stages:**
1. Symbolic Pre-Analysis (Stage 1)
2. LLM Extraction with enriched prompts (Stage 2)
3. Deterministic FOL test compilation (Stage 3)
4. Tiered verification with partial credit
5. SIV Score leaderboard

In [ ]:
# ── Cell 1: Setup & Installation ─────────────────────────────────────────────
import sys, os

# Install dependencies (quiet on Colab; skip locally if already installed)
!pip install -q transformers datasets torch nltk sentencepiece accelerate openai pandas spacy tqdm
!python -m spacy download en_core_web_sm -q

import nltk
for resource in ['wordnet', 'averaged_perceptron_tagger_eng', 'punkt_tab', 'brown']:
    nltk.download(resource, quiet=True)

# Mount Google Drive / add repo root to path
try:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_ROOT = '/content/drive/MyDrive/siv-project'  # adjust if needed
except ImportError:
    REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print(f'Repository root: {REPO_ROOT}')

In [ ]:
# ── Cell 3: Evaluation Configuration ─────────────────────────────────────────
# USE_PARTIAL_CREDIT=False → strict mode (leaderboards/papers): no 0.5 partial credit
# USE_PARTIAL_CREDIT=True  → dense reward mode (model training): 0.5 for CamelCase components
USE_PARTIAL_CREDIT = False

print(f'Partial credit mode: {"ENABLED" if USE_PARTIAL_CREDIT else "DISABLED (strict)"}')

In [ ]:
# ── Cell 2: API key setup ─────────────────────────────────────────────────────
import os

try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY') or ''
except Exception:
    OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', '')

if OPENAI_API_KEY:
    import openai
    client = openai.OpenAI(api_key=OPENAI_API_KEY)
    USE_API = True
    print('OpenAI API available — will use GPT-4o for extraction.')
else:
    client = None
    USE_API = False
    print('No API key — falling back to rule-based extraction.')

In [ ]:
# ── Cell 3: Import SIV modules ────────────────────────────────────────────────
from siv.schema import (
    Entity, EntityType, Fact, MacroTemplate,
    ProblemExtraction, SentenceExtraction,
    UnitTest, TestSuite, VerificationResult,
)
from siv.pre_analyzer import analyze_sentence, format_analyses_for_prompt
from siv.extractor import extract_problem, _fallback_extraction
from siv.compiler import compile_test_suite
from siv.verifier import verify
from siv.scorer import compute_siv_score, score_candidates, aggregate_scores, macro_average
from siv.fol_utils import is_valid_fol, extract_predicates, NLTK_AVAILABLE
from siv.vampire_interface import is_vampire_available, setup_vampire

print(f'NLTK FOL parser available: {NLTK_AVAILABLE}')
print(f'Vampire prover available:  {is_vampire_available()}')

In [ ]:
# ── Cell 4: (Optional) Install Vampire ────────────────────────────────────────
# Uncomment if running on Colab and Vampire is not yet installed:
# from siv.vampire_interface import setup_vampire
# setup_vampire(target_dir=REPO_ROOT)

In [ ]:
# ── Cell 6: Load FOLIO dataset ────────────────────────────────────────────
import json, os
from pathlib import Path

DATA_DIR = Path(REPO_ROOT) / 'data'
folio_path = DATA_DIR / 'folio_problems.json'
folio_problems = []

if folio_path.exists():
    with open(folio_path) as f:
        folio_problems = json.load(f)
    # Detect stale cache written by old code (all IDs were 'unknown')
    unique_ids = {p.get('id') for p in folio_problems}
    if unique_ids <= {'unknown', None}:
        print('Stale cache detected (all IDs are "unknown") — deleting and re-downloading...')
        folio_path.unlink()
        folio_problems = []
    else:
        print(f'Loaded {len(folio_problems)} FOLIO problems from {folio_path}')

if not folio_problems:
    print('folio_problems.json not found — downloading from HuggingFace...')
    try:
        from datasets import load_dataset
        ds = load_dataset('tasksource/folio', split='validation')
        folio_problems = []
        for i, row in enumerate(ds):
            premises_raw = row.get('premises', '')
            if isinstance(premises_raw, str):
                premises_list = [s.strip() for s in premises_raw.splitlines() if s.strip()]
            else:
                premises_list = list(premises_raw)

            # Extract per-premise FOL translations (field name varies by dataset version)
            premises_fol_raw = row.get('premises-FOL', row.get('premises_fol', ''))
            if isinstance(premises_fol_raw, str):
                premises_fol_list = [s.strip() for s in premises_fol_raw.splitlines() if s.strip()]
            else:
                premises_fol_list = list(premises_fol_raw) if premises_fol_raw else []

            folio_problems.append({
                'id':             str(row.get('example_id', row.get('story_id', i))),
                'premises':       premises_list,
                'premises_fol':   premises_fol_list,
                'hypothesis':     row.get('conclusion', row.get('hypothesis', '')),
                'hypothesis_fol': row.get('conclusion-FOL', row.get('hypothesis_fol', '')),
                'label':          row.get('label', ''),
            })
        folio_problems = folio_problems[:100]
        DATA_DIR.mkdir(exist_ok=True)
        with open(folio_path, 'w') as f:
            json.dump(folio_problems, f, indent=2)
        print(f'Saved {len(folio_problems)} problems to {folio_path}')
    except Exception as e:
        print(f'HuggingFace download failed ({e}).\nUsing built-in demo problems.')
        folio_problems = [
            {
                'id': 'demo_001',
                'premises': ['All dogs are animals.', 'Fido is a dog.'],
                'premises_fol': ['all x.(Dog(x) -> Animal(x))', 'Dog(fido)'],
                'hypothesis': 'Fido is an animal.',
                'hypothesis_fol': 'Animal(fido)',
                'label': 'True',
            },
            {
                'id': 'demo_002',
                'premises': ['All cats are mammals.', 'Some mammals are small.'],
                'premises_fol': ['all x.(Cat(x) -> Mammal(x))', 'exists x.(Mammal(x) & Small(x))'],
                'hypothesis': 'Some cats are small.',
                'hypothesis_fol': 'exists x.(Cat(x) & Small(x))',
                'label': 'Unknown',
            },
            {
                'id': 'demo_003',
                'premises': ['No reptiles are warm-blooded.', 'All snakes are reptiles.'],
                'premises_fol': ['all x.(Reptile(x) -> -WarmBlooded(x))', 'all x.(Snake(x) -> Reptile(x))'],
                'hypothesis': 'No snakes are warm-blooded.',
                'hypothesis_fol': 'all x.(Snake(x) -> -WarmBlooded(x))',
                'label': 'True',
            },
        ]


In [ ]:
# ── Cell 6: Stage 1 Demo — Pre-Analysis ──────────────────────────────────────
demo_sentences = [
    'The tall red tree grows quickly.',
    'A Harvard student passed the exam.',
    'Lana Wilson directed After Tiller.',
    'All professional athletes train daily.',
]

print('=' * 60)
print('STAGE 1: Symbolic Pre-Analysis')
print('=' * 60)
for sent in demo_sentences:
    analyses = analyze_sentence(sent)
    print(f'\nSentence: "{sent}"')
    if analyses:
        for ca in analyses:
            print(f'  {ca.modifier!r} + {ca.noun!r}: {ca.recommendation} ({ca.reason})')
    else:
        print('  (no modifier+noun compounds detected)')

In [ ]:
# ── Cell 7: Stage 2 Demo — LLM / Fallback Extraction ─────────────────────────
print('=' * 60)
print('STAGE 2: Entity + Fact Extraction')
print('=' * 60)

for sent in demo_sentences[:3]:
    analyses = analyze_sentence(sent)
    extraction = _fallback_extraction(sent, analyses)   # use fallback for demo
    print(f'\nSentence: "{sent}"')
    print(f'  Macro template: {extraction.macro_template.value}')
    print(f'  Entities: {[(e.id, e.surface, e.entity_type.value) for e in extraction.entities]}')
    print(f'  Facts:    {[(f.pred, f.args, f.negated) for f in extraction.facts]}')

In [ ]:
# ── Cell 8: Stage 3 Demo — Compile Unit Tests ─────────────────────────────────
from siv.extractor import extract_problem as ep

demo_problem_sentences = [
    'The crimson car is running.',
    'All cars have engines.',
]

problem_extraction = ep(
    demo_problem_sentences,
    client=client,
    use_api=USE_API,
    problem_id='demo_car',
)

test_suite = compile_test_suite(problem_extraction)

print('=' * 60)
print('STAGE 3: Compiled Unit Tests')
print('=' * 60)
print(f'\nTotal tests: {test_suite.total_tests} '
      f'({len(test_suite.positive_tests)} recall + '
      f'{len(test_suite.negative_tests)} precision)')

print('\n[+] RECALL tests (candidate must entail):')
for t in test_suite.positive_tests:
    valid = is_valid_fol(t.fol_string)
    print(f'  [{t.test_type:12s}] {t.fol_string}  {"✓" if valid else "✗ INVALID"}')

print('\n[-] PRECISION tests (candidate must NOT entail):')
for t in test_suite.negative_tests:
    valid = is_valid_fol(t.fol_string)
    print(f'  [{t.test_type:12s}] {t.fol_string}  {"✓" if valid else "✗ INVALID"}')

In [ ]:
# ── Cell 9: Candidate Evaluation Demo ────────────────────────────────────────
candidates = [
    'exists x.(Car(x) & Crimson(x) & Running(x))',   # good
    'exists x.Car(x)',                                # missing properties
    'all x.(Car(x) -> CrimsonRunning(x))',            # slugged predicate
    'exists x Car(x',                                 # invalid syntax
]

print('=' * 60)
print('VERIFICATION: Scoring Candidates')
print('=' * 60)

scored = score_candidates(candidates, test_suite)
for rank, cs in enumerate(scored, 1):
    print(f'\n#{rank}: "{cs.candidate_fol[:70]}"')
    print(f'  SIV={cs.siv_score:.3f}  '
          f'recall={cs.recall_rate:.2f}  '
          f'precision={cs.precision_rate:.2f}  '
          f'syntax={cs.syntax_valid}')

In [ ]:
# ── Cell 10: T5 Candidate Generation (requires pretrained model) ──────────────
# This cell is intentionally a no-op. To use T5, copy the commented block
# below into a new cell and configure a checkpoint path.
print('(T5 generation cell — copy the commented code to a new cell to use.)')

# from transformers import T5ForConditionalGeneration, T5Tokenizer
# import torch
#
# MODEL_NAME = 'google/flan-t5-base'   # or your MALLS warmup checkpoint
# tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
# model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)
# model.eval()
#
# def generate_candidates(sentence, num_beams=5, max_new_tokens=128):
#     input_ids = tokenizer(sentence, return_tensors='pt').input_ids
#     with torch.no_grad():
#         outputs = model.generate(
#             input_ids,
#             num_beams=num_beams,
#             num_return_sequences=num_beams,
#             max_new_tokens=max_new_tokens,
#         )
#     return [tokenizer.decode(o, skip_special_tokens=True) for o in outputs]
#
# for sent in demo_problem_sentences:
#     candidates = generate_candidates(sent)
#     scores = score_candidates(candidates, test_suite)
#     print(f'\n"{sent}"  → best SIV: {scores[0].siv_score:.3f}')


In [ ]:
# ── Cell 12: Full FOLIO Evaluation Loop ─────────────────────────────────────
import pandas as pd
from tqdm.auto import tqdm
from siv.compiler import compile_sentence_test_suite
from siv.scorer import aggregate_sentence_scores
from siv.pre_analyzer import analyze_sentence as _analyze
from siv.extractor import extract_sentence as _extract_sent

N_PROBLEMS = 10
USE_API_EVAL = USE_API
MAX_WORKERS  = 5

EVAL_PROBLEMS = folio_problems[:N_PROBLEMS]
# problem_results: pid → List[VerificationResult]  (one per premise with gold FOL)
problem_results = {}

for i, prob in enumerate(tqdm(EVAL_PROBLEMS, desc='Evaluating FOLIO problems')):
    pid = prob.get('id') or f'problem_{i:04d}'
    premises = prob.get('premises', [])
    if isinstance(premises, str):
        premises = [s.strip() for s in premises.splitlines() if s.strip()]
    if not premises:
        continue

    premises_fol = prob.get('premises_fol', [])
    if isinstance(premises_fol, str):
        premises_fol = [s.strip() for s in premises_fol.splitlines() if s.strip()]

    # Pad or trim premises_fol to match premises length
    premises_fol = list(premises_fol) + [''] * max(0, len(premises) - len(premises_fol))

    sentence_results = []
    for j, (premise, gold_fol) in enumerate(zip(premises, premises_fol)):
        if not gold_fol:
            continue
        # Stage 1 + 2: extract single premise sentence
        analyses = _analyze(premise)
        sent_extraction = _extract_sent(premise, analyses, client=client,
                                        model='gpt-4o', use_api=USE_API_EVAL)
        # Stage 3: compile per-sentence test suite
        sent_suite = compile_sentence_test_suite(sent_extraction, f'{pid}_s{j}')
        # Score gold premise FOL against its own test suite
        result = verify(gold_fol, sent_suite, strict_mode=not USE_PARTIAL_CREDIT)
        sentence_results.append(result)

    if sentence_results:
        problem_results[pid] = sentence_results

print(f'Evaluated {len(problem_results)} problems.')


In [ ]:
# ── Cell 13: Diagnostic — per-sentence SIV breakdown for first problem ───────
from siv.verifier import _tier1_vocabulary
from siv.vampire_interface import check_entailment, is_vampire_available
from siv.compiler import compile_sentence_test_suite
from siv.scorer import aggregate_sentence_scores

if problem_results:
    first_pid = next(iter(problem_results))
    first_prob = next(
        (p for p in EVAL_PROBLEMS
         if (p.get('id') or f'problem_{list(EVAL_PROBLEMS).index(p):04d}') == first_pid),
        EVAL_PROBLEMS[0]
    )

    print('=' * 65)
    print(f'DIAGNOSTIC: Problem {first_pid}')
    print('=' * 65)
    print(f'\nPremises:')
    for s in first_prob['premises']:
        print(f'  \u2022 {s}')

    premises = first_prob.get('premises', [])
    premises_fol = first_prob.get('premises_fol', [])
    if isinstance(premises_fol, str):
        premises_fol = [s.strip() for s in premises_fol.splitlines() if s.strip()]
    premises_fol = list(premises_fol) + [''] * max(0, len(premises) - len(premises_fol))

    vamp = is_vampire_available()

    for j, (premise, gold_fol) in enumerate(zip(premises, premises_fol)):
        if not gold_fol:
            continue
        print(f'\n{"─" * 60}')
        print(f'Premise {j+1}: "{premise}"')
        print(f'Gold FOL:  {gold_fol}')

        analyses = _analyze(premise)
        sent_ext = _extract_sent(premise, analyses, client=client,
                                 model='gpt-4o', use_api=USE_API_EVAL)
        print(f'  template = {sent_ext.macro_template.value}')
        for c in sent_ext.constants:
            print(f'  constant: {c.id} = {c.surface!r}')
        for e in sent_ext.entities:
            print(f'  entity:   {e.id} = {e.surface!r} ({e.entity_type.value})')
        for fct in sent_ext.facts:
            neg = '\u00ac' if fct.negated else ' '
            print(f'  fact:     {neg}{fct.pred}({" , ".join(fct.args)})')

        sent_suite = compile_sentence_test_suite(sent_ext, f'{first_pid}_s{j}')
        n_pos = len(sent_suite.positive_tests)
        n_neg = len(sent_suite.negative_tests)
        print(f'  Tests: {sent_suite.total_tests} ({n_pos} recall + {n_neg} precision)')

        print(f'  [+] RECALL tests — gold_fol must entail:')
        for t in sent_suite.positive_tests:
            _, credit = _tier1_vocabulary(gold_fol, t)
            if credit > 0 and vamp:
                proved = check_entailment(gold_fol, t.fol_string, timeout=5)
            else:
                proved = None
            if proved is True:
                status = '\u2713 PASS'
            elif credit > 0 and proved is None:
                status = '~ UNRESOLVED'
            elif credit > 0:
                status = '\u00bd PARTIAL'
            else:
                status = '\u2717 FAIL'
            print(f'    {status:12s} credit={credit:.1f}  [{t.test_type:12s}]  {t.fol_string}')

        result = verify(gold_fol, sent_suite, strict_mode=not USE_PARTIAL_CREDIT)
        print(f'  Score: SIV={result.siv_score:.3f}  '
              f'recall={result.recall_rate:.3f}  '
              f'precision={result.precision_rate:.3f}')

    sent_results = problem_results.get(first_pid, [])
    if sent_results:
        agg = aggregate_sentence_scores(sent_results)
        print(f'\n{"-" * 65}')
        print(f'Problem aggregate ({len(sent_results)} premises scored):')
        print(f'  SIV={agg["siv"]:.3f}  '
              f'recall={agg["recall"]:.3f}  '
              f'precision={agg["precision"]:.3f}')
    print('=' * 65)
else:
    print('No results yet — run Cell 12 first.')


In [ ]:
# ── Cell 14: Results Leaderboard ────────────────────────────────────────────
from siv.scorer import aggregate_sentence_scores

rows = []
for pid, sent_results in problem_results.items():
    if not sent_results:
        continue
    agg = aggregate_sentence_scores(sent_results)
    rows.append({
        'problem_id':   pid,
        'siv_score':    round(agg['siv'],       3),
        'recall':       round(agg['recall'],    3),
        'precision':    round(agg['precision'], 3),
        'n_sentences':  len(sent_results),
    })

df = pd.DataFrame(rows) if rows else pd.DataFrame(
    columns=['problem_id', 'siv_score', 'recall', 'precision', 'n_sentences']
)
print(df.to_string(index=False))

if rows:
    macro_siv  = sum(r['siv_score'] for r in rows) / len(rows)
    macro_rec  = sum(r['recall']    for r in rows) / len(rows)
    macro_prec = sum(r['precision'] for r in rows) / len(rows)
    print(f'\nMacro-average  SIV={macro_siv:.3f}  '
          f'recall={macro_rec:.3f}  '
          f'precision={macro_prec:.3f}')


In [ ]:
# ── Cell 15: Export Results ──────────────────────────────────────────────────
out_path = Path(REPO_ROOT) / 'results_siv.csv'
df.to_csv(out_path, index=False)
print(f'Results saved to {out_path}')
